# This notebook is used to make cord_df dataset

#### The results of this study show that NMF is best suited for finding topics in the corpus, while DTM is best suited for finding topic trends over time. Specifically, NMF was found to have the best performance in identifying clear and distinct topics, while DTM was found to be particularly effective in capturing the evolution of topics over time.



## Also checkout Notebooks for:-

### NMF(Non-Negative Matrix Factorization) - https://www.kaggle.com/code/divyapatel4/topic-modelling-and-visualization-using-nmf/
### LDA (Latent Dirichlet Allocation) - https://www.kaggle.com/code/divyapatel4/topic-modelling-and-visualization-using-lda/
### DTM (Dynamic Topic Model) - https://www.kaggle.com/code/divyapatel4/topic-trends-using-dtm-dynamic-topic-models/


## Pre-processing and cleaning of data is done in the following notebook which is used to make cord_df dataset used in this notebook.

### Data Pre-processing and Cleaning - https://www.kaggle.com/code/divyapatel4/cord-19-dataset-cleaner/




# Kindly upvote if you like my work and feel free to comment and suggest changes. It motivates me to do more work. ##

=================================================================================================================


# Importing Libraries

In [ ]:
! pip install langdetect
import numpy as np
import pandas as pd 
import glob
import json
import seaborn as sns
from tqdm import tqdm
from langdetect import detect
from langdetect import DetectorFactory

import matplotlib.pyplot as plt
plt.style.use('ggplot')

# Loading Data

In [ ]:
root_path = '/kaggle/input/CORD-19-research-challenge/'
metadata_path = f'{root_path}/metadata.csv'
meta_df = pd.read_csv(metadata_path, dtype={
    'pubmed_id': str,
    'Microsoft Academic Paper ID': str, 
    'doi': str
})
meta_df.head()

In [ ]:
meta_df.info()

### Fetching All The json Files

In [ ]:
all_json = ! ls $root_path/document_parses/pdf_json
len(all_json)

In [ ]:
all_json[:5]

### Setting Up The Path

In [ ]:
all_json = [root_path + "document_parses/pdf_json/" + s for s in all_json]
all_json[:5]

# Fetching Data From json Files

In [ ]:
class FileReader:
    def __init__(self, file_path):
        with open(file_path) as file:
            content = json.load(file)
            self.paper_id = content['paper_id']
            self.abstract = []
            self.body_text = []
            # Abstract
            for entry in content['abstract']:
                self.abstract.append(entry['text'])
            # Body text
            for entry in content['body_text']:
                self.body_text.append(entry['text'])
            self.abstract = '\n'.join(self.abstract)
            self.body_text = '\n'.join(self.body_text)
    def __repr__(self):
        return f'{self.paper_id}: {self.abstract[:200]}... {self.body_text[:200]}...'
first_row = FileReader(all_json[0])
print(first_row)

In [ ]:
from tqdm import tqdm
all_json_clean = list()
for idx, entry in tqdm(enumerate(all_json), total=len(all_json)):
    
    try:
        content = FileReader(entry)
    except Exception as e:
        continue 
    
    if len(content.body_text) == 0:
        continue

    all_json_clean.append(all_json[idx])
    
all_json = all_json_clean

len(all_json)


# Randomly Taking Sample From Dataset

In [ ]:
import random

random.seed(42)

all_json = random.sample(all_json, 35000)

In [ ]:
# all_json[:5]

Helper function adds a break after every word when the character length reaches a certain amount. This is for the interactive plot so that the hover tool fits the screen.

In [ ]:
def get_breaks(content, length):
    data = ""
    words = content.split(' ')
    total_chars = 0

    # add break every length characters
    for i in range(len(words)):
        total_chars += len(words[i])
        if total_chars > length:
            data = data + "<br>" + words[i]
            total_chars = 0
        else:
            data = data + " " + words[i]
    return data

# Load The Data Into Dataframes

In [ ]:
!pip install tqdm 

In [ ]:
from tqdm import tqdm

dict_ = {'paper_id': [], 'doi':[], 'abstract': [], 'body_text': [], 'authors': [], 'title': [], 'journal': [], 'abstract_summary': []}
for idx, entry in tqdm(enumerate(all_json), total = len(all_json)):
    
    try:
        content = FileReader(entry)
    except Exception as e:
        continue  # invalid paper format, skip
    
    # get metadata information
    meta_data = meta_df.loc[meta_df['sha'] == content.paper_id]
    # no metadata, skip this paper
    if len(meta_data) == 0:
        continue
    if len(content.body_text) == 0:
        continue
    dict_['abstract'].append(content.abstract)
    dict_['paper_id'].append(content.paper_id)
    dict_['body_text'].append(content.body_text)
    
    # also create a column for the summary of abstract to be used in a plot
    if len(content.abstract) == 0: 
        # no abstract provided
        dict_['abstract_summary'].append("Not provided.")
    elif len(content.abstract.split(' ')) > 100:
        # abstract provided is too long for plot, take first 300 words append with ...
        info = content.abstract.split(' ')[:100]
        summary = get_breaks(' '.join(info), 40)
        dict_['abstract_summary'].append(summary + "...")
    else:
        # abstract is short enough
        summary = get_breaks(content.abstract, 40)
        dict_['abstract_summary'].append(summary)
        
    # get metadata information
    meta_data = meta_df.loc[meta_df['sha'] == content.paper_id]
    
    try:
        # if more than one author
        authors = meta_data['authors'].values[0].split(';')
        if len(authors) > 2:
            # more than 2 authors, may be problem when plotting, so take first 2 append with ...
            dict_['authors'].append(get_breaks('. '.join(authors), 40))
        else:
            # authors will fit in plot
            dict_['authors'].append(". ".join(authors))
    except Exception as e:
        # if only one author - or Null valie
        dict_['authors'].append(meta_data['authors'].values[0])
    
    # add the title information, add breaks when needed
    try:
        title = get_breaks(meta_data['title'].values[0], 40)
        dict_['title'].append(title)
    # if title was not provided
    except Exception as e:
        dict_['title'].append(meta_data['title'].values[0])
    
    # add the journal information
    dict_['journal'].append(meta_data['journal'].values[0])
    
    # add doi
    dict_['doi'].append(meta_data['doi'].values[0])
    
df_covid = pd.DataFrame(dict_, columns=['paper_id', 'doi', 'abstract', 'body_text', 'authors', 'title', 'journal', 'abstract_summary'])
df_covid.head()

In [ ]:
df_covid.info()

# Mark :::: Start after importing from here 

In [ ]:
# df = df_covid.sample(25000, random_state=42)
# # del df_covid

# # export df to csv for later use
# df.to_csv('df_covid_full.csv', index=False)

# from IPython.display import FileLink
# FileLink(r'df_covid.csv')

df = pd.read_csv('/kaggle/input/cord-df/df_20000.csv')
df.head()

Deleting the rows which have null entries

In [ ]:
df.info()


## Handling Multiple Languages

In [ ]:
# set seed
DetectorFactory.seed = 0

# hold label - language
languages = []

# go through each text
for ii in tqdm(range(0,len(df))):
    # split by space into list, take the first x intex, join with space
    text = df.iloc[ii]['body_text'].split(" ")
    
    lang = "en"
    try:
        if len(text) > 50:
            lang = detect(" ".join(text[:50]))
        elif len(text) > 0:
            lang = detect(" ".join(text[:len(text)]))
    # ught... beginning of the document was not in a good format
    except Exception as e:
        all_words = set(text)
        try:
            lang = detect(" ".join(all_words))
        # what!! :( let's see if we can find any text in abstract...
        except Exception as e:
            
            try:
                # let's try to label it through the abstract then
                lang = detect(df.iloc[ii]['abstract_summary'])
            except Exception as e:
                lang = "unknown"
                pass
    
    # get the language    
    languages.append(lang)

Getting the data for each languages

In [ ]:
from pprint import pprint

languages_dict = {}
for lang in set(languages):
    languages_dict[lang] = languages.count(lang)
    
print("Total: {}\n".format(len(languages)))
pprint(languages_dict)

Getting analysis...

In [ ]:
df['language'] = languages
plt.bar(range(len(languages_dict)), list(languages_dict.values()), align='center')
plt.xticks(range(len(languages_dict)), list(languages_dict.keys()))
plt.title("Distribution of Languages in Dataset")
plt.show()

Deleting any article which has not language as English...

In [ ]:
df = df[df['language'] == 'en'] 
df.info()

#### Add The Date, Month and Year Information into the dataframe...

In [ ]:
df10000 = df.sample(10000, random_state=42)
# df10000 = df.sample(10000, random_state=42)

df_copy = df
df = df10000

In [ ]:
df= df_copy

# drop all rows with missing values in 'doi' column
df.dropna(subset=['doi'], inplace=True)


df.info()

In [ ]:
# add date of publishing in the dataset by joining meta_df and df_covid 

# create temporary df of metadata having only doi and publish_time columns

meta_df2 = meta_df[['doi', 'publish_time']]

# Join table on doi column
df = pd.merge(df, meta_df2, on='doi', how='left')

# Convert datatype of publish_time column to datetime
df['publish_time'] = pd.to_datetime(df['publish_time'])

# Sort values by publish_time
df = df.sort_values(by='publish_time')

# Remove all articles published before 2019 November 
df = df[df['publish_time'] >= '2019-11-01']

In [ ]:
df.head()

# plot graph of number of articles for each month in year from november 2019, to december 2022
number_of_articles = df.groupby(df['publish_time'].dt.to_period('M')).size()
number_of_articles.plot(kind='bar', figsize=(20, 10), title='Number of Articles Published Each Month')


In [ ]:


# take 20,000 random sample from df and save it as csv
df_20000 = df.sample(n=20000, random_state=42)
df_20000.to_csv('df_20000.csv', index=False)

# take 15,000 random sample from df and save it as csv
df_15000 = df.sample(n=15000, random_state=42)
df_15000.to_csv('df_15000.csv', index=False)

# take 10,000 random sample from df and save it as csv
df_10000 = df.sample(n=10000, random_state=42)
df_10000.to_csv('df_10000.csv', index=False)



# Analysis About Words in Data

In [ ]:
def word_count(text):
    return len(str(text).split(' '))

In [ ]:
df['word_count'] = df['body_text'].apply(word_count)
df['word_count'].describe()

**Analysis**

In [ ]:
fig = plt.figure(figsize=(10,5))

plt.hist(
    df['word_count'],
    bins=100,
    color='#60505C'
)

plt.title('Distribution - Article Word Count', fontsize=16)
plt.ylabel('Frequency')
plt.xlabel('Word Count')
plt.yticks(np.arange(0, 400, 50))
plt.xticks(np.arange(0, 31000, 5000))

file_name = 'hist'

plt.show()

# Pre-processing The Data

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import string
import re
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import TweetTokenizer, RegexpTokenizer
import nltk

In [ ]:
import nltk
nltk.download('omw-1.4')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer 
lemmatizer = WordNetLemmatizer()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
def vectorize(text, maxx_features, ngram_from, ngram_to):
    
    vectorizer = TfidfVectorizer(max_features=maxx_features, ngram_range=(ngram_from,ngram_to),min_df=0.05,max_df=0.80)
    X = vectorizer.fit_transform(text)
    return X,vectorizer

In [ ]:
# Contraction map
c_dict = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "I would",
    "i'd've": "I would have",
    "i'll": "I will",
    "i'll've": "I will have",
    "i'm": "I am",
    "i've": "I have",
    "isn't": "is not",
    "it'd": "it had",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there had",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we had",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'alls": "you alls",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you had",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
}

# Compiling the contraction dict
c_re = re.compile('(%s)' % '|'.join(c_dict.keys()))

# List of stop words
add_stop = ['said', 'say', '...', 'like', 'cnn', 'ad','et','al','et.','al.']
stop_words = ENGLISH_STOP_WORDS.union(add_stop)
custom_stop_words = [
    'doi', 'preprint', 'copyright', 'peer', 'reviewed', 'org', 'https', 'et', 'al', 'author', 'figure', 
    'rights', 'reserved', 'permission', 'used', 'using', 'biorxiv', 'medrxiv', 'license', 'fig', 'fig.', 
    'al.', 'Elsevier', 'PMC', 'CZI', 'www', 'http', 'No.' , 'no.', 'et.', 'al.'
]
for w in custom_stop_words:
    if w not in stopwords:
        stopwords.append(w)

        
punc = list(set(string.punctuation))

def casual_tokenizer(text):
    tokenizer = TweetTokenizer()
    tokens = tokenizer.tokenize(text)
    return tokens


def expandContractions(text, c_re=c_re):
    def replace(match):
        return c_dict[match.group(0)]
    return c_re.sub(replace, text)


def process_text(text):
    text = casual_tokenizer(text)
    text = [each.lower() for each in text]
    text = [re.sub('[0-9]+', '', each) for each in text]
    text = [expandContractions(each, c_re=c_re) for each in text]
#     text = [SnowballStemmer('english').stem(each) for each in text]
    text = [lemmatizer.lemmatize(each) for each in text]
    text = [w for w in text if w not in punc]
    text = [w for w in text if w not in stop_words]
    text = [each for each in text if len(each) > 1]
    text = [each for each in text if ' ' not in each]
    return text

In [ ]:
# Process the text
df['processed_text'] = df['body_text'].apply(process_text)
df.head()

# Process the text and show proress bar
# for index, row in df.iterrows():
#     df.loc[index, 'processed_text'] = process_text(row['body_text'])
#     pbar.update(1)
# pbar.close()

In [ ]:
text_1 = df['processed_text'].values
text = []
for x in text_1:
    text.append(' '.join(x))

In [ ]:
max_features = 2**12

X2, vectorizer2 = vectorize(text, max_features, 2, 2)  # bigrams

# Print most frequent bigrams and tri-grams
count_values = X2.toarray().sum(axis=0)
vocab = vectorizer2.vocabulary_
df_bigram = pd.DataFrame(sorted([(count_values[i], k) for k, i in vocab.items()], reverse=True)
                         ).rename(columns={0: 'frequency', 1: 'bigrams'})

df_bigram.head(60)




In [ ]:
df_bigram.iloc[60:100]

In [ ]:
X3, vectorizer3 = vectorize(text, max_features, 3, 3)  # trigrams

count_values = X3.toarray().sum(axis=0)
vocab = vectorizer3.vocabulary_
df_trigram = pd.DataFrame(sorted([(count_values[i], k) for k, i in vocab.items()], reverse=True)
                            ).rename(columns={0: 'frequency', 1: 'trigrams'})

df_trigram.head(100)


In [ ]:
# 4-gram 
X4, vectorizer4 = vectorize(text, max_features, 4, 4)  # 4-grams

count_values = X4.toarray().sum(axis=0)
vocab = vectorizer4.vocabulary_
df_4gram = pd.DataFrame(sorted([(count_values[i], k) for k, i in vocab.items()], reverse=True)
                            ).rename(columns={0: 'frequency', 1: '4-grams'})
df_4gram.head(100)

In [ ]:
# 5-gram
X5, vectorizer5 = vectorize(text, max_features, 5, 5)  # 5-grams

count_values = X5.toarray().sum(axis=0)
vocab = vectorizer5.vocabulary_
df_5gram = pd.DataFrame(sorted([(count_values[i], k) for k, i in vocab.items()], reverse=True)
                            ).rename(columns={0: 'frequency', 1: '5-grams'})
df_5gram.head(100)

In [ ]:
# 6-gram
X6, vectorizer6 = vectorize(text, max_features, 6, 6)  # 6-grams

count_values = X6.toarray().sum(axis=0)
vocab = vectorizer6.vocabulary_
df_6gram = pd.DataFrame(sorted([(count_values[i], k) for k, i in vocab.items()], reverse=True)
                            ).rename(columns={0: 'frequency', 1: '6-grams'})
df_6gram.head(100)

 Bigrams 

 mental health,
 public health,
 Health care,
 Long term ,
 Follow up ,
 Immune response ,
 Risk factor ,
 Amino acid ,
 Social medium , 
 Rt pcr ,
 United state ,
 Real time ,
 Well being 
 Social distancing ,
 Sample size ,
 Sars cov ,
 Machine learning ,
 Age group ,
 Physical activity 
 Respiratory syndrome 
 Standard deviation 
 Healthcare worker 
 Nucleic acid 
 Meta analysis 
 Logistic regression 
 Gene expression 
 Important role 


 Trigrams 

 World Health Organization 
 Face to face 
 Intensive care unit 
 polymerase chain reaction
 personal protective equipment
 mann whitney test
 respiratory distress syndrome
 acute respiratory distress
 chi square test
 body mass index
 fisher exact test



 4-gram  

 Severe acute respiratory syndrome 
 acute respiratory syndrome coronavirus
 syndrome coronavirus sars cov
 respiratory syndrome coronavirus sars
 informed consent wa obtained
 intensive care unit icu
 center disease control prevention 
 acute respiratory distress syndrome
 angiotensin converting enzyme ace
 middle east respiratory syndrome


 5-gram
 severe acute respiratory syndrome coronavirus 
 acute respiratory distress syndrome ards


 6-gram 
 — None — 

 7-gram 
 Severe acute respiratory syndrome coronavirus sars cov

In [ ]:
#  Create dictionary of n-grams from above text in small caps
bigrams = {'mental health': 1, 'public health': 1, 'health care': 1, 'long term': 1, 'follow up': 1, 'immune response': 1, 'risk factor': 1, 'amino acid': 1, 'social medium': 1, 'rt pcr': 1, 'united state': 1, 'real time': 1, 'well being': 1, 'social distancing': 1, 'sample size': 1, 'sars cov': 1, 'machine learning': 1, 'age group': 1, 'physical activity': 1, 
           'respiratory syndrome': 1, 'standard deviation': 1, 'healthcare worker': 1, 'nucleic acid': 1, 'meta analysis': 1, 'logistic regression': 1, 'gene expression': 1, 'important role': 1, 'world health organization': 1, 'face to face': 1, 'intensive care unit': 1, 'polymerase chain reaction': 1, 'personal protective equipment': 1, 'mann whitney test': 1, 
           'respiratory distress syndrome': 1, 'acute respiratory distress': 1, 'chi square test': 1, 'body mass index': 1, 'fisher exact test': 1, 'severe acute respiratory syndrome': 1, 'acute respiratory syndrome coronavirus': 1, 'syndrome coronavirus sars cov': 1, 'respiratory syndrome coronavirus sars': 1, 'informed consent wa obtained': 1, 'intensive care unit icu': 1, 
           'center disease control prevention': 1, 'acute respiratory distress syndrome': 1, 'angiotensin converting enzyme ace': 1, 'middle east respiratory syndrome': 1, 'severe acute respiratory syndrome coronavirus sars cov': 1, 'acute respiratory distress syndrome ards': 1, 'close contact':1, 'community transmission':1, 'community spread':1, 'contact tracing':1, 
           'cordon sanitaire':1, 'droplet transmission':1, 'droplet spread':1, 'elective surgery':1, 'elective surgeries':1, 'essential activities':1, 'essential services':1, 'essential travel':1, 'exposure risk':1, 'face mask':1, 'face masks':1, 'face shield':1, 'face shields':1, 'face covering':1, 'face coverings':1, 'flattening curve':1, 'home isolation':1, 'n95 mask':1, 
           'n95 masks':1, 'n95 respirator':1, 'n95 respirators':1, 'reproductive rate':1, 'social distancing':1, 'viral shedding':1, 'viral load':1, 'virus transmission':1}

trigrams = {'world health organization': 1, 'face to face': 1, 'intensive care unit': 1, 'polymerase chain reaction': 1, 'personal protective equipment': 1,
            'mann whitney test': 1, 'respiratory distress syndrome': 1, 'acute respiratory distress': 1, 'chi square test': 1, 'body mass index': 1, 'fisher exact test': 1, 
            'case fatality rate':1, 'drive through testing':1, 'essential government functions':1, 'face covering mandate':1, 'flattening the curve':1, 'negative pressure rooms':1, 'shelter in place':1}

fourgrams = {'severe acute respiratory syndrome': 1, 'acute respiratory syndrome coronavirus': 1, 'syndrome coronavirus sars cov': 1, 'respiratory syndrome coronavirus sars': 1, 'informed consent wa obtained': 1,
             'intensive care unit icu': 1, 'center disease control prevention': 1, 'acute respiratory distress syndrome': 1, 'angiotensin converting enzyme ace': 1, 'middle east respiratory syndrome': 1}

fivegrams = {'severe acute respiratory syndrome coronavirus': 1,
             'acute respiratory distress syndrome ards': 1}

sevengrams = {'severe acute respiratory syndrome coronavirus sars cov': 1}


In [ ]:
# Merge all tokens in df['processed_text'] to one string
df['temp'] = df['processed_text'].apply(lambda x: ' '.join(x))

# If two consecutive words like mental and health are encountered in the text, then replace them with mental_health if they are in bigrams dictionary
def replace_bigrams(text):
    text = text.split()
    # for words in text check if word and next word are in bigrams dictionary, if yes then replace them with bigram
    for i in range(len(text)-1):
        if text[i] + ' ' + text[i+1] in bigrams:
            text[i] = text[i] + '_' + text[i+1]
            text[i+1] = ''
            
    # remove empty strings
    text = [x for x in text if x != '']
    return text



# Same as above but for trigrams
def replace_trigrams(text):
    text = text.split()
    for i in range(len(text)-2):
        if text[i] + ' ' + text[i+1] + ' ' + text[i+2] in trigrams:
            text[i] = text[i] + '_' + text[i+1] + '_' + text[i+2]
            text[i+1] = ''
            text[i+2] = ''
            
    text = [x for x in text if x != '']
    return text


def replace_fourgrams(text):
    text = text.split()
    for i in range(len(text)-3):
        if text[i] + ' ' + text[i+1] + ' ' + text[i+2] + ' ' + text[i+3] in fourgrams:
            text[i] = text[i] + '_' + text[i+1] + '_' + text[i+2] + '_' + text[i+3]
            text[i+1] = ''
            text[i+2] = ''
            text[i+3] = ''
            
    text = [x for x in text if x != '']
    return text

def replace_fivegrams(text):
    text = text.split()
    for i in range(len(text)-4):
        if text[i] + ' ' + text[i+1] + ' ' + text[i+2] + ' ' + text[i+3] + ' ' + text[i+4] in fivegrams:
            text[i] = text[i] + '_' + text[i+1] + '_' + text[i+2] + '_' + text[i+3] + '_' + text[i+4]
            text[i+1] = ''
            text[i+2] = ''
            text[i+3] = ''
            text[i+4] = ''
            
    text = [x for x in text if x != '']
    return text

def replace_sevengrams(text):
    text = text.split()
    for i in range(len(text)-6):
        if text[i] + ' ' + text[i+1] + ' ' + text[i+2] + ' ' + text[i+3] + ' ' + text[i+4] + ' ' + text[i+5] + ' ' + text[i+6] in sevengrams:
            text[i] = text[i] + '_' + text[i+1] + '_' + text[i+2] + '_' + text[i+3] + '_' + text[i+4] + '_' + text[i+5] + '_' + text[i+6]
            text[i+1] = ''
            text[i+2] = ''
            text[i+3] = ''
            text[i+4] = ''
            text[i+5] = ''
            text[i+6] = ''
            
    text = [x for x in text if x != '']
    return text


df['temp'] = df['temp'].apply(lambda x: replace_sevengrams(x))
df['temp'] = df['temp'].apply(lambda x: ' '.join(x))
df['temp'] = df['temp'].apply(lambda x: replace_fivegrams(x))
df['temp'] = df['temp'].apply(lambda x: ' '.join(x))
df['temp'] = df['temp'].apply(lambda x: replace_fourgrams(x))
df['temp'] = df['temp'].apply(lambda x: ' '.join(x))
df['temp'] = df['temp'].apply(lambda x: replace_trigrams(x))
df['temp'] = df['temp'].apply(lambda x: ' '.join(x))
df['temp'] = df['temp'].apply(lambda x: replace_bigrams(x))
df['temp'] = df['temp'].apply(lambda x: ' '.join(x))



In [ ]:
# df.to_csv('processed_text.csv', index=False)
df.head(100)


In [ ]:
df['processed_text'] = df['temp']
df.drop('temp', axis=1, inplace=True)

# Tokenize df['processed_text']
df['processed_text'] = df['processed_text'].apply(lambda x: x.split())


In [ ]:
print('Frequency of severe_acute_respiratory_syndrome_coronavirus_sars_cov in df[\'processed_text\'] array of tokens: ', df['processed_text'].apply(lambda x: 'severe_acute_respiratory_syndrome_coronavirus_sars_cov' in x).sum())
print('Frequency of intensive_care_unit_icu in df[\'processed_text\'] array of tokens: ', df['processed_text'].apply(lambda x: 'intensive_care_unit_icu' in x).sum())


## Analysis About Top Words After Pre-processing

In [ ]:
from collections import Counter

In [ ]:
# Get the top 20 most common words among all the articles
p_text = df['processed_text']

# Flaten the list of lists
p_text = [item for sublist in p_text for item in sublist]

# Top 20
top_20 = pd.DataFrame(
    Counter(p_text).most_common(100),
    columns=['word', 'frequency']
)

top_20

In [ ]:
# Plot a bar chart for the top 20 most frequently occuring words
fig = plt.figure(figsize=(20,7))

g = sns.barplot(
    x='word',
    y='frequency',
    data=top_20,
    palette='GnBu_d'
)

g.set_xticklabels(
    g.get_xticklabels(),
    rotation=90,
    fontsize=10
)

plt.yticks(fontsize=14)
plt.xlabel('Words', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Top 20 Words', fontsize=17)

plt.show()


## Number of Unique Words...

In [ ]:
# Get the number of unique words after processing
num_unique_words = len(set(p_text))
num_unique_words

# Finding The Number Of Best Topics As Parameter

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from gensim.models.nmf import Nmf
from operator import itemgetter

In [ ]:
# Use Gensim's NMF to get the best num of topics via coherence score
texts = df['processed_text']

# Create a dictionary
# In gensim a dictionary is a mapping between words and their integer id
dictionary = Dictionary(texts)

# Filter out extremes to limit the number of features
dictionary.filter_extremes(
    no_below=3,
    no_above=0.85,
    keep_n=int(0.7*num_unique_words)
)

# Create the bag-of-words format (list of (token_id, token_count))
corpus = [dictionary.doc2bow(text) for text in texts]

# Create a list of the topic numbers we want to try
topic_nums = list(np.arange(3, 52, 3))

# Run the nmf model and calculate the coherence score
# for each number of topics
coherence_scores = []

for num in tqdm(topic_nums):
    nmf = Nmf(
        corpus=corpus,
        num_topics=num,
        id2word=dictionary,
        chunksize=int(0.7*0.6*num_unique_words),
        passes=5,
        kappa=.1,
        minimum_probability=0.01,
        w_max_iter=300,
        w_stop_condition=0.0001,
        h_max_iter=100,
        h_stop_condition=0.001,
        eval_every=10,
        normalize=True,
        random_state=42
    )
    
    # Run the coherence model to get the score
    cm = CoherenceModel(
        model=nmf,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    coherence_scores.append(round(cm.get_coherence(), 5))

# Get the number of topics with the highest coherence score
scores = list(zip(topic_nums, coherence_scores))
best_num_topics = sorted(scores, key=itemgetter(1), reverse=True)[0][0]

# Plot the results
fig = plt.figure(figsize=(15, 7))

plt.plot(
    topic_nums,
    coherence_scores,
    linewidth=3,
    color='#4287f5'
)

plt.xlabel("Topic Num", fontsize=14)
plt.ylabel("Coherence Score", fontsize=14)
plt.title('Coherence Score by Topic Number - Best Number of Topics: {}'.format(best_num_topics), fontsize=18)
plt.xticks(np.arange(5, max(topic_nums) + 1, 5), fontsize=12)
plt.yticks(fontsize=12)
plt.show()

In [ ]:
# Funtion to remove duplicate words
def unique_words(text): 
    ulist = []
    [ulist.append(x) for x in text if x not in ulist]
    return ulist

def whitespace_tokenizer(text): 
    pattern = r"(?u)\b\w\w+\b" 
    tokenizer_regex = RegexpTokenizer(pattern)
    tokens = tokenizer_regex.tokenize(text)
    return tokens

def top_words(topic, n_top_words):
    return topic.argsort()[:-n_top_words - 1:-1]  

def topic_table(model, feature_names, n_top_words):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        t = (topic_idx)
        topics[t] = [feature_names[i] for i in top_words(topic, n_top_words)]
    return pd.DataFrame(topics)

In [ ]:
# Now use the number of topics with the 
# highest coherence score to run the 
# sklearn nmf model

texts = df['processed_text']

# Create the tfidf weights
tfidf_vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.75,
    max_features=int(0.7*num_unique_words),
    ngram_range=(1, 2),
    preprocessor=' '.join
)

tfidf = tfidf_vectorizer.fit_transform(texts)

# Save the feature names for later to create topic summaries
tfidf_fn = tfidf_vectorizer.get_feature_names()

# Run the nmf model
nmf = NMF(
    n_components=best_num_topics,
    init='nndsvd',
    max_iter=500,
    l1_ratio=0.0,
    solver='cd',
    alpha=0.0,
    tol=1e-4,
    random_state=42
).fit(tfidf)

In [ ]:
# Use the top words for each cluster by tfidf weight
# to create 'topics'

# Getting a df with each topic by document
docweights = nmf.transform(tfidf_vectorizer.transform(texts))

n_top_words = 10

topic_df = topic_table(
    nmf,
    tfidf_fn,
    n_top_words
).T

# Cleaning up the top words to create topic summaries
topic_df['topics'] = topic_df.apply(lambda x: [' '.join(x)], axis=1) # Joining each word into a list
topic_df['topics'] = topic_df['topics'].str[0]  # Removing the list brackets
topic_df['topics'] = topic_df['topics'].apply(lambda x: whitespace_tokenizer(x)) # tokenize
topic_df['topics'] = topic_df['topics'].apply(lambda x: unique_words(x))  # Removing duplicate words
topic_df['topics'] = topic_df['topics'].apply(lambda x: [' '.join(x)])  # Joining each word into a list
topic_df['topics'] = topic_df['topics'].str[0]  # Removing the list brackets

topic_df.head()